# Reestructuración de Datos (Reshaping)

In [1]:
import polars as pl

## DataFrames Anchos vs. Altos
- Ancho y alto describen dos formas de organizar datos en una tabla.
- Una variable es un atributo de datos que puede tener múltiples valores.
- Los `DataFrames` anchos almacenan la misma variable en múltiples columnas.
- Los DataFrames anchos se expanden horizontalmente: su número de columnas crece.
- Los `DataFrames` altos almacenan cada variable en una sola columna.
- Los `DataFrames` altos se expanden verticalmente: su número de filas crece.

### Ejemplo
- El dataset `wide_store_sales.csv` es un dataset ancho.
- Almacena el mismo atributo/variable (ingresos) en múltiples columnas (Jan, Feb, etc).
- A medida que lleguen más valores de ingresos para meses futuros, el `DataFrame` se expandirá en ancho.

In [2]:
pl.read_csv("wide_store_sales.csv")

store_id,location,Jan,Feb,Mar,Apr,May,Jun
i64,str,i64,i64,i64,i64,i64,i64
101,"""Central""",12000,12500,13000,13500,14000,14500
102,"""North""",15000,14500,16000,15500,16500,17000
103,"""East""",10000,9500,11000,10500,11500,12000
104,"""South""",18000,19000,20000,21000,22000,23000
105,"""West""",13000,13500,14000,14500,15000,15500


## El Método `unpivot` para Convertir un DataFrame Ancho a uno Largo

### ¿Cómo funciona `unpivot`?

Imagina que tienes una tabla donde cada mes es una columna:
| store_id | Jan | Feb |
|----------|-----|-----|
| 101      | 120 | 130 |

Al aplicar `unpivot`, conviertes esas columnas en filas:
| store_id | variable | value |
|----------|----------|-------|
| 101      | Jan      | 120   |
| 101      | Feb      | 130   |

**Parámetros clave:**
- `index`: Las columnas que se mantienen fijas (identificadores).
- `on`: Las columnas que quieres 'desenrollar' (si se omite, usa todas las demás).
- `variable_name`: Nombre para la nueva columna que contendrá los nombres de las antiguas columnas.
- `value_name`: Nombre para la nueva columna que contendrá los datos numéricos.

In [3]:
sales = pl.read_csv("wide_store_sales.csv")
sales

store_id,location,Jan,Feb,Mar,Apr,May,Jun
i64,str,i64,i64,i64,i64,i64,i64
101,"""Central""",12000,12500,13000,13500,14000,14500
102,"""North""",15000,14500,16000,15500,16500,17000
103,"""East""",10000,9500,11000,10500,11500,12000
104,"""South""",18000,19000,20000,21000,22000,23000
105,"""West""",13000,13500,14000,14500,15000,15500


In [ ]:
sales.unpivot(
    index=["store_id", "location"], on=["Jan", "Feb", "Mar", "Apr", "May", "Jun"]
)

store_id,location,variable,value
i64,str,str,i64
101,"""Central""","""Jan""",12000
102,"""North""","""Jan""",15000
103,"""East""","""Jan""",10000
104,"""South""","""Jan""",18000
105,"""West""","""Jan""",13000
…,…,…,…
101,"""Central""","""Jun""",14500
102,"""North""","""Jun""",17000
103,"""East""","""Jun""",12000


In [4]:
# Cuando se omite `on` todas las otras columnas se emplean por unpivot
sales.unpivot(index=["store_id", "location"])

store_id,location,variable,value
i64,str,str,i64
101,"""Central""","""Jan""",12000
102,"""North""","""Jan""",15000
103,"""East""","""Jan""",10000
104,"""South""","""Jan""",18000
105,"""West""","""Jan""",13000
…,…,…,…
101,"""Central""","""Jun""",14500
102,"""North""","""Jun""",17000
103,"""East""","""Jun""",12000


- Usa el parámetro `variable_name` para renombrar la columna de variables (la que contendrá los nombres de las columnas anteriores).
- Usa el parámetro `value_name` para renombrar la columna de valores (la que contendrá los valores de las celdas).

In [5]:
sales.unpivot(index=["store_id", "location"], variable_name="month", value_name="sales")

store_id,location,month,sales
i64,str,str,i64
101,"""Central""","""Jan""",12000
102,"""North""","""Jan""",15000
103,"""East""","""Jan""",10000
104,"""South""","""Jan""",18000
105,"""West""","""Jan""",13000
…,…,…,…
101,"""Central""","""Jun""",14500
102,"""North""","""Jun""",17000
103,"""East""","""Jun""",12000


### Lectura Adicional
- https://docs.pola.rs/user-guide/transformations/unpivot/
- https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.unpivot.html

## El Método `pivot` para Convertir un DataFrame Alto a uno Ancho

### ¿Cómo funciona `pivot`?

Imagina que tienes datos de calificaciones en formato largo:
| student | subject | score |
|---------|---------|-------|
| Alice   | math    | 88    |
| Alice   | history | 52    |

Al aplicar `pivot`, las materias se convierten en columnas:
| student | math | history |
|---------|------|---------|
| Alice   | 88   | 52      |

**Parámetros clave:**
- `index`: La columna que identifica las filas (ej. el nombre del estudiante).
- `on`: La columna cuyos valores únicos se convertirán en los nombres de las nuevas columnas (ej. las materias).
- `values`: La columna que contiene los datos que rellenarán las celdas (ej. las notas).
- `aggregate_function`: (Opcional) La función para resumir datos si existen duplicados.

In [6]:
grades = pl.read_csv("student_grades.csv")
grades

student,subject,score
str,str,i64
"""Alice""","""math""",88
"""Bob""","""math""",76
"""Charlie""","""math""",90
"""Diana""","""math""",67
"""Ethan""","""math""",95
…,…,…
"""Alice""","""writing""",100
"""Bob""","""writing""",50
"""Charlie""","""writing""",82


- Una vista más ancha puede facilitar el análisis del rendimiento de un estudiante en todas las materias.
- El método `pivot` transforma un `DataFrame` largo en uno ancho.


In [ ]:
grades.pivot(index="student", on="subject", values="score")

student,math,history,writing
str,i64,i64,i64
"""Alice""",88,52,100
"""Bob""",76,69,50
"""Charlie""",90,13,82
"""Diana""",67,26,67
"""Ethan""",95,100,null


### Lectura Adicional
- https://docs.pola.rs/user-guide/transformations/pivot/#eager
- https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.pivot.html

## Tablas Dinámicas I (Pivot Tables)
- El `DataFrame` `student_grades_expanded` tiene múltiples entradas para un estudiante y una calificación.

In [7]:
grades = pl.read_csv("student_grades_expanded.csv")
grades.head(3)

student,subject,score
str,str,i64
"""Alice""","""math""",88
"""Bob""","""math""",76
"""Charlie""","""math""",90


- El dataset almacena las calificaciones de los estudiantes durante 2 años en la escuela.
- Por lo tanto, ciertas combinaciones de estudiante y materia pueden aparecer dos veces.

In [8]:
grades.filter(pl.col("student") == "Alice")

student,subject,score
str,str,i64
"""Alice""","""math""",88
"""Alice""","""history""",52
"""Alice""","""writing""",100
"""Alice""","""math""",92
"""Alice""","""history""",59
"""Alice""","""writing""",16


- Un método `pivot` regular falla debido a valores duplicados en la columna `student`.
- Cada combinación de nombre de estudiante y materia aparece dos veces (una por año), por lo que Polars no puede elegir una sola calificación por combinación.

In [10]:
# grades.pivot(index="student", on="subject", values="score")

- El parámetro `aggregate_function` establece el algoritmo para seleccionar el valor en cada combinación duplicada.
- Un argumento de `first` selecciona la primera ocurrencia de cada valor único.
- En este ejemplo, Polars usará la calificación del examen para la primera ocurrencia de cada nombre de estudiante + materia.
- Este dataset representa las calificaciones del primer año.

In [11]:
grades.pivot(index="student", on="subject", values="score", aggregate_function="first")

student,math,history,writing
str,i64,i64,i64
"""Alice""",88,52,100
"""Bob""",76,69,50
"""Charlie""",90,13,82
"""Diana""",67,26,67
"""Ethan""",95,100,null


- Un argumento de `last` selecciona la última ocurrencia de cada valor único.
- Este dataset representa las calificaciones del segundo año.

In [12]:
grades.pivot(index="student", on="subject", values="score", aggregate_function="last")

student,math,history,writing
str,i64,i64,i64
"""Alice""",92,59,16
"""Bob""",100,62,25
"""Charlie""",38,null,88
"""Diana""",42,98,72
"""Ethan""",13,null,30


- Existen argumentos complementarios `max` y `min` para elegir el valor más grande o más pequeño en cada coincidencia.

In [13]:
grades.pivot(index="student", on="subject", values="score", aggregate_function="max")

student,math,history,writing
str,i64,i64,i64
"""Alice""",92,59,100
"""Bob""",100,69,50
"""Charlie""",90,13,88
"""Diana""",67,98,72
"""Ethan""",95,100,30


In [14]:
grades.pivot(index="student", on="subject", values="score", aggregate_function="min")

student,math,history,writing
str,i64,i64,i64
"""Alice""",88,52,16
"""Bob""",76,62,25
"""Charlie""",38,13,82
"""Diana""",42,26,67
"""Ethan""",13,100,30


### Lectura Adicional
- https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.pivot.html

## Tablas Dinámicas II (Pivot Tables)
- Las funciones de agregación de la lección anterior (`first`, `last`, `max` y `min`) eligieron un valor de un conjunto de valores posibles.
- Otras funciones pueden realizar operaciones de agregación sobre todos los valores dentro de cada combinación.

In [ ]:
grades = pl.read_csv("student_grades_expanded.csv")
grades.head(3)

student,subject,score
str,str,i64
"""Alice""","""math""",88
"""Bob""","""math""",76
"""Charlie""","""math""",90


- La agregación `sum` suma los valores.

In [ ]:
grades.pivot(index="student", on="subject", values="score", aggregate_function="sum")

student,math,history,writing
str,i64,i64,i64
"""Alice""",180,111,116
"""Bob""",176,131,75
"""Charlie""",128,13,170
"""Diana""",109,124,139
"""Ethan""",108,100,30


- La agregación `mean` calcula el promedio de los valores.

In [ ]:
grades.pivot(index="student", on="subject", values="score", aggregate_function="mean")

student,math,history,writing
str,f64,f64,f64
"""Alice""",90.0,55.5,58.0
"""Bob""",88.0,65.5,37.5
"""Charlie""",64.0,13.0,85.0
"""Diana""",54.5,62.0,69.5
"""Ethan""",54.0,100.0,30.0


- La función de agregación `len` cuenta el número de valores en cada grupo.

In [ ]:
grades.pivot(index="student", on="subject", values="score", aggregate_function="len")

student,math,history,writing
str,u32,u32,u32
"""Alice""",2,2,2
"""Bob""",2,2,2
"""Charlie""",2,2,2
"""Diana""",2,2,2
"""Ethan""",2,2,2


- Una ventaja de las tablas dinámicas es que podemos ver los datos desde diferentes ángulos.
- Intercambiemos los ejes: materias como valores de fila, estudiantes como columnas.

### Lectura Adicional
- https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.pivot.html

### Lectura Adicional
- https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.transpose.html

---
# Ejercicios de Práctica

### Ejercicio 1
Dado el siguiente `DataFrame` de temperaturas promedio por ciudad y trimestre, conviértelo de formato ancho a alto. Las columnas de trimestre deben quedar en una columna llamada `"trimestre"` y los valores en una columna llamada `"temperatura"`.

In [ ]:
temperaturas = pl.DataFrame({
    "ciudad": ["La Paz", "Cochabamba", "Sucre"],
    "Q1": [8, 11, 12],
    "Q2": [20, 19, 25],
    "Q3": [32, 27, 36],
    "Q4": [10, 13, 14],
})
temperaturas

ciudad,Q1,Q2,Q3,Q4
str,i64,i64,i64,i64
"""La Paz""",8,20,32,10
"""Cochabamba""",11,19,27,13
"""Sucre""",12,25,36,14


### Ejercicio 2
Dado el siguiente `DataFrame` alto de ventas por producto y región, transfórmalo a formato ancho donde cada región sea una columna y los valores sean las ventas.

In [ ]:
ventas = pl.DataFrame({
    "producto": ["Laptop", "Laptop", "Laptop", "Tablet", "Tablet", "Tablet"],
    "region": ["Norte", "Centro", "Sur", "Norte", "Centro", "Sur"],
    "ventas": [150, 200, 120, 80, 95, 110],
})
ventas

producto,region,ventas
str,str,i64
"""Laptop""","""Norte""",150
"""Laptop""","""Centro""",200
"""Laptop""","""Sur""",120
"""Tablet""","""Norte""",80
"""Tablet""","""Centro""",95
"""Tablet""","""Sur""",110


### Ejercicio 3
El siguiente `DataFrame` contiene las calificaciones de empleados en evaluaciones trimestrales (algunos empleados tienen más de una evaluación por trimestre). Crea una tabla dinámica que muestre el **promedio** de calificación por empleado y trimestre.

In [ ]:
evaluaciones = pl.DataFrame({
    "empleado": ["Ana", "Ana", "Ana", "Luis", "Luis", "Luis", "Ana", "Luis"],
    "trimestre": ["Q1", "Q2", "Q3", "Q1", "Q2", "Q3", "Q1", "Q1"],
    "calificacion": [85, 90, 88, 78, 82, 80, 92, 75],
})
evaluaciones

empleado,trimestre,calificacion
str,str,i64
"""Ana""","""Q1""",85
"""Ana""","""Q2""",90
"""Ana""","""Q3""",88
"""Luis""","""Q1""",78
"""Luis""","""Q2""",82
"""Luis""","""Q3""",80
"""Ana""","""Q1""",92
"""Luis""","""Q1""",75


### Ejercicio 4
Dado el siguiente `DataFrame` ancho de gastos mensuales por departamento:
1. Primero, conviértelo a formato largo (columnas: `"departamento"`, `"mes"`, `"gasto"`).
2. Luego, crea una tabla donde las filas sean los meses y las columnas los departamentos, mostrando la **suma** de los gastos.

In [ ]:
gastos = pl.DataFrame({
    "departamento": ["Ventas", "Marketing", "Ingeniería"],
    "Enero": [5000, 8000, 12000],
    "Febrero": [5500, 7500, 13000],
    "Marzo": [6000, 9000, 11500],
})
gastos

departamento,Enero,Febrero,Marzo
str,i64,i64,i64
"""Ventas""",5000,5500,6000
"""Marketing""",8000,7500,9000
"""Ingeniería""",12000,13000,11500


### Ejercicio 5
Dado el siguiente `DataFrame` de precios de productos en diferentes sucursales, crea una tabla dinámica que muestre el **precio máximo** por `categoria` (filas) y `region` (columnas).

In [ ]:
precios_productos = pl.DataFrame({
    "categoria": ["Electronica", "Electronica", "Hogar", "Hogar", "Electronica", "Hogar"],
    "region": ["Norte", "Sur", "Norte", "Sur", "Norte", "Sur"],
    "precio": [500, 550, 200, 180, 600, 220]
})
precios_productos

categoria,region,precio
str,str,i64
"""Electronica""","""Norte""",500
"""Electronica""","""Sur""",550
"""Hogar""","""Norte""",200
"""Hogar""","""Sur""",180
"""Electronica""","""Norte""",600
"""Hogar""","""Sur""",220
